# 📈 Etapa 3: Modelo Predictivo de Riesgo de Prórroga en Contratos Públicos
**Observatorio Ciudadano de Contratación Pública (SECOP II)**

En esta etapa construimos y evaluamos un modelo clásico de Machine Learning para estimar, **exclusivamente con la información disponible el día de la firma**, la probabilidad de que un contrato requiera adiciones de tiempo (`dias_adicionados > 0`). Cumplimos con partición temporal estricta, comparación de modelos frente a una línea base, explicabilidad por permutación y exportación del pipeline para consumo en tiempo real en la aplicación web.

In [ ]:
# Recarga automática de módulos y configuración de rutas
%load_ext autoreload
%autoreload 2

import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from observatorio.config import SQLITE_DB_PATH, RANDOM_SEED, MODEL_ARTIFACT_PATH

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", None)
print("✅ Entorno ML configurado.")
print(f"Base de datos contratos: {SQLITE_DB_PATH}")
print(f"Ruta artefacto modelo : {MODEL_ARTIFACT_PATH}")


---
# 📈 PARTE 3: Modelo Predictivo de Riesgo de Prórroga en Contratos

## 3.1 Entendiendo el Problema: ¿Qué Contratos Terminan Pidiendo Más Plazo?
Cuando un contrato público sufre prórrogas (`dias_adicionados > 0`), suele reflejar retos de planeación, cuellos de botella en la ejecución o complejidades propias del sector. Antes de entrenar cualquier algoritmo, exploramos los datos para entender qué tan frecuentes son las prórrogas y cómo varían según el tipo de contrato, la modalidad de selección y el monto involucrado.

In [ ]:
# Construcción y auditoría de la variable objetivo (dias_adicionados > 0)
import sqlite3
import matplotlib.pyplot as plt
from observatorio.ml.features import COLUMNA_OBJETIVO, construir_dataset_modelado

# 1. Cargar los contratos desde la base SQLite construida en la Parte 1
with sqlite3.connect(SQLITE_DB_PATH) as conn:
    df_contratos = pd.read_sql_query("SELECT * FROM contratos", conn)

# 2. Crear objetivo + variables derivadas y quitar contratos que aún no terminan su plazo
df_modelo = construir_dataset_modelado(df_contratos)

print(f"Contratos totales                  : {len(df_contratos):,}")
print(f"Contratos aptos para modelar       : {len(df_modelo):,}  (plazo ya vencido)")
print(f"Excluidos por estar aún en plazo   : {len(df_contratos) - len(df_modelo):,}")
print(f"Tasa de prórroga en datos modelables: {df_modelo[COLUMNA_OBJETIVO].mean():.2%}")

# 3. Gráficos exploratorios
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Distribución de clases
df_modelo[COLUMNA_OBJETIVO].map({0: "Sin prórroga", 1: "Con prórroga"}).value_counts().plot.bar(
    ax=axes[0, 0], color=["steelblue", "darkorange"], title="Distribución de la variable objetivo", rot=0)

# Tasa de prórroga por modalidad (solo modalidades con al menos 50 contratos)
tasa_modalidad = df_modelo.groupby("modalidad_de_contratacion")[COLUMNA_OBJETIVO].agg(["mean", "size"])
tasa_modalidad.query("size >= 50")["mean"].sort_values().plot.barh(
    ax=axes[0, 1], color="darkorange", title="Tasa de prórroga por modalidad")

# Tasa de prórroga por tipo de contrato
tasa_tipo = df_modelo.groupby("tipo_de_contrato")[COLUMNA_OBJETIVO].agg(["mean", "size"])
tasa_tipo.query("size >= 50")["mean"].sort_values().plot.barh(
    ax=axes[1, 0], color="seagreen", title="Tasa de prórroga por tipo de contrato")

# Valor del contrato en escala log según la clase
for clase, color in [(0, "steelblue"), (1, "darkorange")]:
    df_modelo.loc[df_modelo[COLUMNA_OBJETIVO] == clase, "valor_contrato_log"].plot.hist(
        ax=axes[1, 1], bins=40, alpha=0.6, density=True, color=color,
        label="Con prórroga" if clase else "Sin prórroga")
axes[1, 1].set_title("Valor del contrato (log) por clase")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 3.2 Regla de Oro: Usar Solo lo que se Sabe el Día de la Firma
Un error común al modelar datos públicos es incluir columnas que se actualizan meses después de firmado el contrato (como el estado de liquidación o los pagos acumulados), lo que produce modelos artificialmente perfectos pero inútiles en la vida real. Aquí seleccionamos exclusivamente las variables conocidas en el instante de la firma.

In [ ]:
# Listas explícitas de variables conocidas el día de la firma (definidas en src/observatorio/ml/features.py)
from observatorio.ml.features import (
    FEATURES_CATEGORICAS_PERMITIDAS,
    FEATURES_NUMERICAS_PERMITIDAS,
    VARIABLES_EXCLUIDAS_LEAKAGE,
    verificar_ausencia_leakage,
)

FEATURES_MODELO = FEATURES_NUMERICAS_PERMITIDAS + FEATURES_CATEGORICAS_PERMITIDAS
print("Numéricas  :", FEATURES_NUMERICAS_PERMITIDAS)
print("Categóricas:", FEATURES_CATEGORICAS_PERMITIDAS)

# Auditoría de fuga: lanza error si alguna variable post-firma se coló en la lista
verificar_ausencia_leakage(FEATURES_MODELO)
print(f"\n✅ Sin data leakage. Variables excluidas a propósito: {sorted(VARIABLES_EXCLUIDAS_LEAKAGE)}")

# Nulos por variable: se imputan dentro del pipeline (mediana / 'DESCONOCIDO') en lugar de borrar filas
display(df_modelo[FEATURES_MODELO].isna().sum().to_frame("nulos").query("nulos > 0"))

## 3.3 Entrenar con el Pasado, Evaluar en el Futuro
Para simular cómo funcionaría esta herramienta en manos de una veeduría o una oficina de control interno, dividimos los contratos cronológicamente: entrenamos los modelos con el historial previo y los ponemos a prueba sobre los contratos más recientes.

In [ ]:
# Partición temporal: el 80 % más antiguo para entrenar y el 20 % más reciente para probar
from observatorio.ml.pipeline import construir_preprocesador

# df_modelo ya viene ordenado por fecha_de_firma, así que basta con cortar por posición
indice_corte = int(len(df_modelo) * 0.8)
df_train = df_modelo.iloc[:indice_corte]
df_test = df_modelo.iloc[indice_corte:]

X_train, y_train = df_train[FEATURES_MODELO], df_train[COLUMNA_OBJETIVO]
X_test, y_test = df_test[FEATURES_MODELO], df_test[COLUMNA_OBJETIVO]

display(pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Prueba"],
    "Contratos": [len(df_train), len(df_test)],
    "Desde": [df_train["fecha_de_firma"].min().date(), df_test["fecha_de_firma"].min().date()],
    "Hasta": [df_train["fecha_de_firma"].max().date(), df_test["fecha_de_firma"].max().date()],
    "% con prórroga": [y_train.mean() * 100, y_test.mean() * 100],
}).round(2))

# ColumnTransformer: imputación + escalado (numéricas) e imputación + One-Hot (categóricas).
# Va DENTRO de cada pipeline, por lo que se ajusta únicamente con X_train (sin fuga hacia el test).
preprocesador = construir_preprocesador()
print(f"Columnas tras el preprocesamiento: {preprocesador.fit(X_train).transform(X_train.head()).shape[1]}")

## 3.4 Comparación de Modelos: De una Línea Base a Ensambles de Árboles
Comparamos un punto de referencia simple (`Baseline`) frente a distintos modelos clásicos de Machine Learning (**Regresión Logística**, **Random Forest** y **Gradient Boosting**), ajustando el desbalance natural de los datos para identificar cuál logra detectar mejor los contratos propensos a prórroga.

In [ ]:
# Pipelines para Baseline, Regresión Logística, Random Forest y Gradient Boosting
from observatorio.ml.evaluation import entrenar_y_comparar_modelos

# 1. Validación cruzada temporal (PR-AUC) y búsqueda de hiperparámetros
modelos, resultados_cv, nombre_final, modelo_final = entrenar_y_comparar_modelos(X_train, y_train, n_splits=3)

for nom, score in resultados_cv.items():
    print(f"{nom:<30} PR-AUC CV = {score:.3f}")

print(f"\n🏆 Modelo seleccionado: {nombre_final}")

## 3.5 Resultados Reales: ¿Qué Aprendió el Modelo y Dónde se Equivoca?
Más allá de un puntaje global, analizamos qué factores pesan más en la estimación de riesgo (como la duración planeada inicial, la modalidad o el valor del contrato) y en qué tipos de procesos el modelo acierta o falla con mayor frecuencia.

In [ ]:
# Tabla comparativa de métricas en el conjunto de prueba temporal + matrices de confusión y curvas PR
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay
from observatorio.ml.evaluation import (
    analizar_errores_por_modalidad,
    calcular_importancia_variables,
    generar_tabla_comparativa_prueba,
)

# 1. Métricas en prueba temporal (datos no vistos durante el entrenamiento)
df_metricas = generar_tabla_comparativa_prueba(modelos, X_test, y_test, resultados_cv)
display(df_metricas)
print(f"Modelo seleccionado (mejor PR-AUC en validación cruzada): {nombre_final}")

# 2. Matrices de confusión (umbral 0.5) de los modelos con aprendizaje
nombres_graficar = [n for n in modelos if n != "Baseline"]
fig, axes = plt.subplots(1, len(nombres_graficar), figsize=(5 * len(nombres_graficar), 4))
for ax, nombre in zip(axes, nombres_graficar):
    ConfusionMatrixDisplay.from_estimator(
        modelos[nombre], X_test, y_test, ax=ax, colorbar=False,
        display_labels=["Sin prórroga", "Con prórroga"]
    )
    ax.set_title(nombre)
plt.tight_layout()
plt.show()

# 3. Curvas Precisión-Recall (frente al azar = prevalencia)
fig, ax = plt.subplots(figsize=(7, 5))
for nombre, m in modelos.items():
    PrecisionRecallDisplay.from_estimator(m, X_test, y_test, ax=ax, name=nombre)
ax.axhline(y_test.mean(), linestyle="--", color="gray", label="Azar")
ax.set_title("Curvas Precisión-Recall en prueba temporal")
ax.legend()
plt.show()

# 4. Importancia por permutación
serie_imp = calcular_importancia_variables(modelo_final, X_test, y_test)
serie_imp.sort_values().plot.barh(figsize=(7, 5), title=f"Importancia por permutación — {nombre_final}")
plt.xlabel("Caída en PR-AUC")
plt.show()

# 5. Análisis de errores por modalidad de contratación
df_errores = analizar_errores_por_modalidad(modelo_final, df_test, X_test, y_test)
display(df_errores)

---
# 🚀 PARTE 4: Del Análisis a la Aplicación Interactiva

## 4.1 Empaquetado del Modelo para Consulta en Tiempo Real
Exportamos el flujo completo del modelo ganador en un único archivo listo para producción (`pipeline_prorrogas.joblib`) y comprobamos que la aplicación web pueda cargarlo y generar estimaciones en una fracción de segundo.

In [ ]:
# Guardar el artefacto del modelo con joblib.dump() y sus metadatos en JSON
import time
from observatorio.config import MODEL_ARTIFACT_PATH, MODEL_METADATA_PATH
from observatorio.ml.evaluation import guardar_artefacto_y_metadatos
from observatorio.ml.inference import cargar_modelo_prorrogas, predecir_riesgo_prorroga

# 1. Exportar pipeline completo y metadatos JSON auditables
metadatos = guardar_artefacto_y_metadatos(
    modelo=modelo_final,
    nombre_modelo=nombre_final,
    df_train=df_train,
    df_test=df_test,
    df_metricas=df_metricas,
)
print(f"✅ Modelo guardado en   : {MODEL_ARTIFACT_PATH}")
print(f"✅ Metadatos guardados en: {MODEL_METADATA_PATH}")

# 2. Verificación de carga en frío e inferencia unitaria (< 2 segundos, Métricas ML)
modelo_cargado = cargar_modelo_prorrogas()
contrato_ejemplo = X_test.iloc[0].to_dict()

t0 = time.perf_counter()
resultado = predecir_riesgo_prorroga(modelo_cargado, contrato_ejemplo)
latencia_ms = (time.perf_counter() - t0) * 1000

prob_memoria = modelo_final.predict_proba(X_test.iloc[[0]])[0, 1]
print(f"\nPredicción contrato ejemplo: {resultado}")
print(f"Tiempo de respuesta        : {latencia_ms:.1f} ms (requisito < 2000 ms)")
print(f"¿Coincide con memoria?     : {np.isclose(prob_memoria, resultado['probabilidad_prorroga'], atol=1e-4)}")

## 4.2 Transparencia, Alcance y Límites del Modelo
Para cerrar el flujo de desarrollo, dejamos documentadas las condiciones con las que fue entrenado el modelo, para qué sirve y cuáles son sus límites:
- **Para qué sirve:** Funciona como un filtro temprano para que periodistas, veedurías y equipos de auditoría decidan qué contratos revisar primero entre miles de procesos firmados.
- **Qué no debe hacerse con él:** No es un detector de corrupción ni un veredicto sobre la gestión de un contratista o entidad pública.
- **Limitaciones naturales:** Los datos tabulares de SECOP II resumen el contrato pero no contienen el texto detallado de los estudios previos ni las actas adjuntas, por lo que toda revisión debe culminar en el expediente electrónico original.